# 1. Static embeddings

A static embedding assigns one fixed vector to each word, regardless of the sentence in which it appears.

\\[ \text{Word} \longrightarrow \text{One fixed vector} \\]

Examples include Word2Vec, GloVe, and FastText.

Imagine these two sentences:

- “I deposited money at the bank.”
- “I sat beside the river bank.”

With a static embedding, `bank` receives the same pretrained vector in both sentences. The vector does not automatically change based on whether the word refers to a financial institution or a riverbank.

Static embedding example

Financial bank

“Deposited money at the bank”

Same vector

River bank

“Sat beside the river bank”

Same vector

Both occurrences retrieve the same embedding-table row for the token “bank.”

Advantages

- Simple and computationally efficient.
- Can be pretrained on large text corpora.
- Useful for learning basic word representations.

Limitation: One vector cannot directly represent every meaning of an ambiguous word.

# 2. Contextual embeddings

A contextual embedding produces a word representation that depends on the surrounding words.

\\[ \text{Word}+\text{Context}\longrightarrow \text{Context-dependent vector} \\]

Examples include contextual representations from ELMo, BERT, and GPT.

Using the same two sentences:

- “I deposited money at the bank.”
- “I sat beside the river bank.”

The model processes each sentence and generates a representation influenced by its context. Therefore, the two occurrences of `bank` can have different vectors.

Contextual embedding example

“I deposited money at the bank.”

Bank representation A

Influenced by “deposited money”

“I sat beside the river bank.”

Bank representation B

Influenced by “river”

The representations can differ because the surrounding context differs.

Advantages

- Better captures different meanings of the same word.
- Represents relationships between words in a sentence.
- Useful for tasks such as sentiment analysis, question answering, and named entity recognition.

Trade-off: Contextual models generally require more computation than a simple static embedding lookup.

# 3. Key differences

| Feature            | Static                    | Contextual                      |
| ------------------ | ------------------------- | ------------------------------- |
| Vector for a word  | Fixed                     | Depends on context              |
| Ambiguous words    | Same vector per token     | Can have different vectors      |
| Context processing | Not part of the lookup    | Used to produce representations |
| Examples           | Word2Vec, GloVe, FastText | ELMo, BERT, GPT                 |
| Computation        | Generally cheaper         | Generally more expensive        |

## 4. Important distinction: embedding layer vs contextual model

Remember our previous notebook, `12.2_Embedding_Layer.ipynb`?

A standard PyTorch `nn.Embedding` performs a lookup. The same token ID retrieves the same vector regardless of the sentence.

A contextual model, such as BERT, uses a neural network to transform token representations using surrounding context. Its output for the same token can therefore change across sentences.

Also, static does not mean untrainable: static embeddings can be fine-tuned during model training. The term means the representation is not dynamically conditioned on each occurrence's context.

---

In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
# Demonstrate static embeddings

vocab = {
    "<PAD>": 0,
    "bank": 1,
    "money": 2,
    "river": 3
}

embedding_dim = 4

static_embedding = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=embedding_dim
).to(device)

bank_id = torch.tensor([vocab["bank"]], device=device)

vector_1 = static_embedding(bank_id)
vector_2 = static_embedding(bank_id)

print("First vector:", vector_1)
print("Second vector:", vector_2)
print("Identical:", torch.equal(vector_1, vector_2))

First vector: tensor([[-2.4025, -0.8130, -0.0227,  0.8601]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)
Second vector: tensor([[-2.4025, -0.8130, -0.0227,  0.8601]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)
Identical: True


In [4]:
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print("Device:", device)
print("Hidden dimension:", model.config.hidden_size)

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yuvan\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. 

Device: cuda
Hidden dimension: 768


In [5]:
# Compare the same word in two contexts

sentences = [
    "I deposited money at the bank.",
    "I sat beside the river bank."
]

bank_vectors = []

with torch.no_grad():
    for sentence in sentences:
        inputs = tokenizer(
            sentence,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)

        tokens = tokenizer.convert_ids_to_tokens(
            inputs["input_ids"][0]
        )

        # Locate the token "bank"
        bank_position = tokens.index("bank")

        # Extract its contextual representation
        bank_vector = outputs.last_hidden_state[
            0, bank_position, :
        ]

        bank_vectors.append(bank_vector)

        print("\nSentence:", sentence)
        print("Tokens:", tokens)
        print("Bank vector shape:", bank_vector.shape)
        print("First 5 dimensions:", bank_vector[:5])


Sentence: I deposited money at the bank.
Tokens: ['[CLS]', 'i', 'deposited', 'money', 'at', 'the', 'bank', '.', '[SEP]']
Bank vector shape: torch.Size([768])
First 5 dimensions: tensor([ 0.6793, -0.4071, -0.0688,  0.0783,  0.8408], device='cuda:0')

Sentence: I sat beside the river bank.
Tokens: ['[CLS]', 'i', 'sat', 'beside', 'the', 'river', 'bank', '.', '[SEP]']
Bank vector shape: torch.Size([768])
First 5 dimensions: tensor([ 0.1816, -0.5837,  0.0526, -0.2241, -0.4875], device='cuda:0')


In [6]:
# measure the difference 

from torch.nn.functional import cosine_similarity

similarity = cosine_similarity(
    bank_vectors[0].unsqueeze(0),
    bank_vectors[1].unsqueeze(0)
).item()

print("Cosine similarity:", similarity)
print("Vectors identical:", torch.equal(
    bank_vectors[0], bank_vectors[1]
))

Cosine similarity: 0.5411713123321533
Vectors identical: False
